<h2>Description</h2>

L'objectif de ce code est de fusionner toutes les données, donc à savoir les données de la rue de la convention avec les données externes. 
Nous voulons donc créer un dataframe qui contient :
<li>les informations météorologiques</li>
<li>les informations sur les vacances et jour fériés</li>
<li>les informations sur l'opération "Paris respire"</li>

In [216]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

doc = 'convention.csv'

print("Version installée de pandas : ")
print(pd.__version__)

Version installée de pandas : 
2.3.0


<h3>Analyse du dataset relatif à l'axe</h3>

In [217]:
df_axe = pd.read_csv("../datasets_axes_bruts/" + doc, sep=";")

# On convertit la date en format convenable 
df_axe['Date et heure de comptage'] = pd.to_datetime(df_axe['Date et heure de comptage'], 
                                      errors='coerce', utc=True).dt.tz_convert('Europe/Paris').dt.tz_localize(None)  

df_axe.head() 

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,Etat arc,Date debut dispo data,Date fin dispo data,geo_point_2d,geo_shape
0,5671,Convention,2025-09-02 11:00:00,462.0,4.20056,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,Ouvert,1997-01-20,2023-01-01,"48.838634371808965, 2.293205602717535","{""coordinates"": [[2.291878306272707, 48.839238..."
1,5671,Convention,2025-10-09 11:00:00,514.0,3.86722,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,Ouvert,1997-01-20,2023-01-01,"48.838634371808965, 2.293205602717535","{""coordinates"": [[2.291878306272707, 48.839238..."
2,5671,Convention,2025-09-02 12:00:00,482.0,3.24945,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,Ouvert,1997-01-20,2023-01-01,"48.838634371808965, 2.293205602717535","{""coordinates"": [[2.291878306272707, 48.839238..."
3,5671,Convention,2025-09-02 13:00:00,563.0,4.57445,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,Ouvert,1997-01-20,2023-01-01,"48.838634371808965, 2.293205602717535","{""coordinates"": [[2.291878306272707, 48.839238..."
4,5671,Convention,2024-10-04 12:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,Invalide,1997-01-20,2023-01-01,"48.838634371808965, 2.293205602717535","{""coordinates"": [[2.291878306272707, 48.839238..."


In [218]:
df_axe.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval
count,8747.0,8747,4867.000000,4974.000000,8747.0,8747.0
mean,5671.0,2025-04-26 02:38:19.028238336,398.152044,3.609356,2937.0,2973.0
min,5671.0,2024-10-01 05:00:00,31.000000,0.000000,2937.0,2973.0
25%,5671.0,2025-01-22 16:30:00,196.000000,1.319167,2937.0,2973.0
50%,5671.0,2025-04-25 20:00:00,390.000000,2.626945,2937.0,2973.0
75%,5671.0,2025-08-06 21:30:00,577.000000,4.448750,2937.0,2973.0
max,5671.0,2025-11-07 00:00:00,1024.000000,37.348340,2937.0,2973.0
std,0.0,NaN,224.378023,3.771356,0.0,0.0


In [219]:
df_axe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8747 non-null   int64         
 1   Libelle                    8747 non-null   object        
 2   Date et heure de comptage  8747 non-null   datetime64[ns]
 3   Débit horaire              4867 non-null   float64       
 4   Taux d'occupation          4974 non-null   float64       
 5   Etat trafic                8747 non-null   object        
 6   Identifiant noeud amont    8747 non-null   int64         
 7   Libelle noeud amont        8747 non-null   object        
 8   Identifiant noeud aval     8747 non-null   int64         
 9   Libelle noeud aval         8747 non-null   object        
 10  Etat arc                   8747 non-null   object        
 11  Date debut dispo data      8747 non-null   object        
 12  Date f

In [220]:
df_axe['date'] = df_axe['Date et heure de comptage'].dt.date
df_axe['heure'] = df_axe['Date et heure de comptage'].dt.hour
df_axe['dow'] = df_axe['Date et heure de comptage'].dt.dayofweek
df_axe['mois'] = df_axe['Date et heure de comptage'].dt.month
df_axe['annee'] = df_axe['Date et heure de comptage'].dt.year
df_axe['jour_mois'] = df_axe['Date et heure de comptage'].dt.day
jour_map = {0:'Lun',1:'Mar',2:'Mer',3:'Jeu',4:'Ven',5:'Sam',6:'Dim'}
df_axe['jour_semaine'] = df_axe['dow'].map(jour_map)

In [221]:
fig = px.line(
    df_axe.sort_values('Date et heure de comptage'),
    x='Date et heure de comptage', y='Débit horaire',
    title=f"Débit horaire",
    labels={'Date et heure de comptage':'Date/Heure','Débit horaire':'Débit (véh/h)'}
)

fig.show()

In [222]:
profil = (df_axe
          .groupby(['jour_semaine','heure'], as_index=False)['Débit horaire']
          .mean())

fig = px.line(
    profil, x='heure', y='Débit horaire', color='jour_semaine',
    markers=True, title=f"Profil horaire moyen par jour",
    labels={'heure':'Heure','Débit horaire':'Débit moyen (véh/h)','jour_semaine':'Jour'}
)
fig.update_layout(template='simple_white')
fig.show()

In [223]:
heat = (df_axe
        .groupby(['jour_semaine','heure'], as_index=False)['Débit horaire']
        .mean())


heat['jour_semaine'] = pd.Categorical(heat['jour_semaine'],
                                      categories=['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'],
                                      ordered=True)

fig = px.imshow(
    heat.pivot(index='jour_semaine', columns='heure', values='Débit horaire').values,
    labels=dict(x="Heure", y="Jour", color="Débit moyen"),
    x=list(range(24)),
    y=['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'],
    title=f"Carte de chaleur"
)
fig.update_layout(template='simple_white')
fig.show()


In [224]:
df_scatter = df_axe.dropna(subset=['Débit horaire','Taux d\'occupation']).copy()
fig = px.scatter(
    df_scatter, x='Taux d\'occupation', y='Débit horaire',
    color='jour_semaine', opacity=0.7,
    title=f"Débit vs Taux d’occupation",
    labels={'Taux d\'occupation':'Taux d’occupation','Débit horaire':'Débit (véh/h)'}
)

fig.update_layout(template='simple_white')
fig.show()

In [225]:
box = df_axe.dropna(subset=['Débit horaire']).copy()
fig = px.box(
    box, x='jour_semaine', y='Débit horaire', points='all',
    title=f"Distribution du débit par jour",
    labels={'jour_semaine':'Jour','Débit horaire':'Débit (véh/h)'}
)
fig.update_layout(template='simple_white')
fig.show()


In [226]:
etat_jour = (df_axe
             .assign(jour=pd.to_datetime(df_axe['date']))
             .groupby(['jour','Etat trafic'], as_index=False)
             .size())

fig = px.area(
    etat_jour, x='jour', y='size', color='Etat trafic',
    title=f"État du trafic (comptes journaliers)",
    labels={'jour':'Date','size':'Occurrences'}
)
fig.update_layout(template='simple_white', hovermode='x unified')
fig.show()

In [227]:
mensuel = (df_axe
           .set_index('Date et heure de comptage')
           .resample('MS')['Débit horaire']
           .mean()
           .to_frame('debit_moyen')
           .reset_index())

mensuel['trend_3m'] = mensuel['debit_moyen'].rolling(3, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=mensuel['Date et heure de comptage'], y=mensuel['debit_moyen'],
    mode='lines+markers', name='Moyenne mensuelle'
))
fig.add_trace(go.Scatter(
    x=mensuel['Date et heure de comptage'], y=mensuel['trend_3m'],
    mode='lines', name='Tendance (MM-3)', line=dict(width=4)
))

fig.update_layout(
    title="Débit horaire – tendance mensuelle (moyenne + lissage 3 mois)",
    xaxis_title="Mois",
    yaxis_title="Débit moyen (véh/h)",
    template="simple_white",
    hovermode="x unified"
)
fig.show()


In [228]:
profil = (df_axe
          .groupby(['annee','mois'], as_index=False)['Débit horaire']
          .mean()
          .rename(columns={'Débit horaire':'debit_moyen'}))

fig = px.line(
    profil, x='mois', y='debit_moyen', color='annee',
    markers=True,
    title="Profil mensuel du débit par année",
    labels={'mois':'Mois', 'debit_moyen':'Débit moyen (véh/h)', 'annee':'Année'}
)
fig.update_layout(template='simple_white', xaxis=dict(dtick=1))
fig.show()


In [229]:
df_axe['heure_sin'] = np.sin(2 * np.pi * df_axe['heure'] / 24)
df_axe['heure_cos'] = np.cos(2 * np.pi * df_axe['heure'] / 24)

df_axe['jour_sin'] = np.sin(2 * np.pi * df_axe['dow'] / 7)
df_axe['jour_cos'] = np.cos(2 * np.pi * df_axe['dow'] / 7)

df_axe['mois_sin'] = np.sin(2 * np.pi * df_axe['mois'] / 12)
df_axe['mois_cos'] = np.cos(2 * np.pi * df_axe['mois'] / 12)

In [230]:
df_axe.head(10)

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,...,mois,annee,jour_mois,jour_semaine,heure_sin,heure_cos,jour_sin,jour_cos,mois_sin,mois_cos
0,5671,Convention,2025-09-02 11:00:00,462.0,4.20056,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,...,9,2025,2,Mar,2.588190e-01,-0.965926,0.781831,0.623490,-1.000000e+00,-1.836970e-16
1,5671,Convention,2025-10-09 11:00:00,514.0,3.86722,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,...,10,2025,9,Jeu,2.588190e-01,-0.965926,0.433884,-0.900969,-8.660254e-01,5.000000e-01
2,5671,Convention,2025-09-02 12:00:00,482.0,3.24945,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,...,9,2025,2,Mar,1.224647e-16,-1.000000,0.781831,0.623490,-1.000000e+00,-1.836970e-16
3,5671,Convention,2025-09-02 13:00:00,563.0,4.57445,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,...,9,2025,2,Mar,-2.588190e-01,-0.965926,0.781831,0.623490,-1.000000e+00,-1.836970e-16
4,5671,Convention,2024-10-04 12:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,...,10,2024,4,Ven,1.224647e-16,-1.000000,-0.433884,-0.900969,-8.660254e-01,5.000000e-01
5,5671,Convention,2025-02-18 21:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,...,2,2025,18,Mar,-7.071068e-01,0.707107,0.781831,0.623490,8.660254e-01,5.000000e-01
6,5671,Convention,2025-02-18 20:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,...,2,2025,18,Mar,-8.660254e-01,0.500000,0.781831,0.623490,8.660254e-01,5.000000e-01
7,5671,Convention,2025-02-18 19:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,...,2,2025,18,Mar,-9.659258e-01,0.258819,0.781831,0.623490,8.660254e-01,5.000000e-01
8,5671,Convention,2025-02-18 15:00:00,NaN,NaN,Inconnu,2937,Lecourbe-Convention,2973,Convention-Blomet,...,2,2025,18,Mar,-7.071068e-01,-0.707107,0.781831,0.623490,8.660254e-01,5.000000e-01
9,5671,Convention,2024-12-09 01:00:00,157.0,0.77889,Fluide,2937,Lecourbe-Convention,2973,Convention-Blomet,...,12,2024,9,Lun,2.588190e-01,0.965926,0.000000,1.000000,-2.449294e-16,1.000000e+00


<h3>Nous insérons d'abord les données météorologiques</h3>

In [231]:
meteo = pd.read_csv("../datasets_externes_clean/meteo.csv", sep=";")
meteo['datetime'] = pd.to_datetime(meteo['datetime'])

In [232]:
df_merge = pd.merge(df_axe, meteo, left_on="Date et heure de comptage", right_on="datetime", how="left")

In [233]:
df_merge.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,jour_cos,mois_sin,mois_cos,Unnamed: 0,precipitations heure,duree prec (en min),force moyenne vent (m/s),Température,ensoleillement (en min),datetime
count,8747.0,8747,4867.000000,4974.000000,8747.0,8747.0,8747.000000,8747.000000,8747.000000,8747.000000,...,8747.000000,8.747000e+03,8.747000e+03,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747
mean,5671.0,2025-04-26 02:38:19.028238336,398.152044,3.609356,2937.0,2973.0,11.506574,2.994741,6.668000,2024.804047,...,-0.006239,-5.500528e-02,3.106604e-02,11546.638619,0.077867,3.980107,2.933686,13.218383,13.081685,2025-04-26 02:38:19.028238336
min,5671.0,2024-10-01 05:00:00,31.000000,0.000000,2937.0,2973.0,0.000000,0.000000,1.000000,2024.000000,...,-0.900969,-1.000000e+00,-1.000000e+00,6581.000000,0.000000,0.000000,0.000000,-3.600000,0.000000,2024-10-01 05:00:00
25%,5671.0,2025-01-22 16:30:00,196.000000,1.319167,2937.0,2973.0,6.000000,1.000000,4.000000,2025.000000,...,-0.900969,-8.660254e-01,-5.000000e-01,9304.500000,0.000000,0.000000,2.000000,8.700000,0.000000,2025-01-22 16:30:00
50%,5671.0,2025-04-25 20:00:00,390.000000,2.626945,2937.0,2973.0,12.000000,3.000000,7.000000,2025.000000,...,-0.222521,-2.449294e-16,6.123234e-17,11540.000000,0.000000,0.000000,2.800000,13.000000,0.000000,2025-04-25 20:00:00
75%,5671.0,2025-08-06 21:30:00,577.000000,4.448750,2937.0,2973.0,17.000000,5.000000,10.000000,2025.000000,...,0.623490,5.000000e-01,5.000000e-01,14013.500000,0.000000,0.000000,3.700000,17.700000,20.000000,2025-08-06 21:30:00
max,5671.0,2025-11-07 00:00:00,1024.000000,37.348340,2937.0,2973.0,23.000000,6.000000,12.000000,2025.000000,...,1.000000,1.000000e+00,1.000000e+00,16224.000000,14.300000,60.000000,9.300000,37.600000,60.000000,2025-11-07 00:00:00
std,0.0,NaN,224.378023,3.771356,0.0,0.0,6.919751,1.993350,3.434265,0.396955,...,0.707910,7.313334e-01,6.791720e-01,2769.396136,0.518787,12.900969,1.311747,6.678668,21.956183,NaN


In [234]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8747 non-null   int64         
 1   Libelle                    8747 non-null   object        
 2   Date et heure de comptage  8747 non-null   datetime64[ns]
 3   Débit horaire              4867 non-null   float64       
 4   Taux d'occupation          4974 non-null   float64       
 5   Etat trafic                8747 non-null   object        
 6   Identifiant noeud amont    8747 non-null   int64         
 7   Libelle noeud amont        8747 non-null   object        
 8   Identifiant noeud aval     8747 non-null   int64         
 9   Libelle noeud aval         8747 non-null   object        
 10  Etat arc                   8747 non-null   object        
 11  Date debut dispo data      8747 non-null   object        
 12  Date f

In [235]:
df_axe = df_merge

<h3>Nous allons maintenant ajouter les données relatives au jours piétonnisés</h3>

Nous allons donc ajouter une colonne de 1 ou 0 pour indiquer si oui ou non il s'agissait d'un jour piéton.

In [236]:
pieton = pd.read_csv("../datasets_externes_clean/pieton.csv", sep=";")
pieton.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  20 non-null     int64 
 1   date_debut  20 non-null     object
 2   date_fin    20 non-null     object
dtypes: int64(1), object(2)
memory usage: 608.0+ bytes


In [237]:
pieton['date_debut'] = pd.to_datetime(pieton['date_debut'])
pieton['date_fin'] = pd.to_datetime(pieton['date_fin'])

Nous écrivons ci-dessous une fonction pour déterminer si pour une date donnée, certains axes parisiens étaient piétonnisés.

In [238]:
def est_pietonnise(date):
    debut = pieton['date_debut']
    fin = pieton['date_fin']
    for i in range (len(debut)):
        if date >= debut[i] and date < fin[i]:
            return 1
    return 0

df_axe['est_pieton'] = df_axe['Date et heure de comptage'].apply(est_pietonnise)

In [239]:
df_axe[df_axe['est_pieton'] == 1][['Date et heure de comptage', 'est_pieton']]

,Date et heure de comptage,est_pieton
233,2024-11-03 16:00:00,1
234,2024-11-03 15:00:00,1
242,2024-11-03 14:00:00,1
243,2024-11-03 13:00:00,1
244,2024-11-03 12:00:00,1
...,...,...
6885,2025-09-21 15:00:00,1
6886,2025-09-21 12:00:00,1
6887,2025-09-21 08:00:00,1
6888,2025-09-21 07:00:00,1


In [240]:
df_axe = df_axe.sort_values(by="Date et heure de comptage", ascending=True)

<h3>On va maintenant ajouter des données sur les vacances, jours fériés...</h3>

In [241]:
vacances = pd.read_csv("../datasets_externes_clean/vacances.csv", sep=";")
vacances ['Date de début'] = pd.to_datetime(vacances['Date de début'])
vacances ['Date de fin'] = pd.to_datetime(vacances['Date de fin'])

In [242]:
def est_jour_vacances(dat):
    date_sans_heure = dat.date()
    debut = vacances['Date de début'].dt.date
    fin = vacances['Date de fin'].dt.date
    for i in range (len(debut)):
        if date_sans_heure >= debut[i] and date_sans_heure < fin[i]:
            return 1
    return 0

df_axe['est_vacances'] = df_axe['Date et heure de comptage'].apply(est_jour_vacances)

In [243]:
df = df_axe[df_axe['est_vacances'] == 1][['Date et heure de comptage', 'est_vacances']]

In [244]:
df.head(50)

,Date et heure de comptage,est_vacances
4694,2024-10-19 00:00:00,1
5569,2024-10-19 01:00:00,1
5568,2024-10-19 02:00:00,1
5567,2024-10-19 03:00:00,1
6217,2024-10-19 04:00:00,1
5566,2024-10-19 05:00:00,1
5565,2024-10-19 06:00:00,1
5564,2024-10-19 07:00:00,1
5563,2024-10-19 08:00:00,1
6216,2024-10-19 09:00:00,1


In [245]:
def est_avant_vacances(date):
    date_sans_heure = date.date()
    debut = vacances['Date de début'].dt.date
    for i in range (len(debut)):
        if date_sans_heure == debut[i] - pd.Timedelta(days=1):
            return 1
    return 0

df_axe['est_avant_vacances'] = df_axe['Date et heure de comptage'].apply(est_avant_vacances)

In [246]:
df_axe[df_axe['est_avant_vacances'] == 1][['est_avant_vacances', 'Date et heure de comptage']]

,est_avant_vacances,Date et heure de comptage
6337,1,2024-10-18 00:00:00
6149,1,2024-10-18 01:00:00
5860,1,2024-10-18 02:00:00
6148,1,2024-10-18 03:00:00
5859,1,2024-10-18 04:00:00
...,...,...
4142,1,2025-10-17 19:00:00
4143,1,2025-10-17 20:00:00
3939,1,2025-10-17 21:00:00
4144,1,2025-10-17 22:00:00


In [247]:
ferie = pd.read_csv("../datasets_externes_clean/ferie.csv", sep=";")

ferie['Date de début'] = pd.to_datetime(ferie['Date de début'], format='%d/%m/%Y')

In [248]:
def est_ferie(date):
    date = date.date()
    j_ferie = ferie['Date de début'].dt.date
    for el in j_ferie :
        if date == el:
            return 1
    return 0

df_axe['est_ferie'] = df_axe['Date et heure de comptage'].apply(est_ferie)

In [249]:
df_axe[df_axe['est_ferie']==1]['Date et heure de comptage']

8719   2024-11-01 00:00:00
8585   2024-11-01 01:00:00
8746   2024-11-01 02:00:00
8745   2024-11-01 03:00:00
3111   2024-11-01 04:00:00
               ...        
418    2025-11-01 19:00:00
22     2025-11-01 20:00:00
419    2025-11-01 21:00:00
23     2025-11-01 22:00:00
420    2025-11-01 23:00:00
Name: Date et heure de comptage, Length: 264, dtype: datetime64[ns]

In [250]:
def est_avant_ferie(date):
    date = date.date()
    j_ferie = ferie['Date de début'].dt.date
    for el in j_ferie :
        if date == el - pd.Timedelta(days=1):
            return 1
    return 0

df_axe['est_avant_ferie'] = df_axe['Date et heure de comptage'].apply(est_avant_ferie)

In [251]:
df_axe[df_axe['est_avant_ferie']==1]['Date et heure de comptage']

8615   2024-10-31 00:00:00
8739   2024-10-31 01:00:00
8738   2024-10-31 02:00:00
8714   2024-10-31 03:00:00
8713   2024-10-31 04:00:00
               ...        
8625   2025-10-31 19:00:00
8727   2025-10-31 20:00:00
8624   2025-10-31 21:00:00
8623   2025-10-31 22:00:00
8726   2025-10-31 23:00:00
Name: Date et heure de comptage, Length: 264, dtype: datetime64[ns]

In [252]:
def est_rentree(date):
    date = date.date()  
    rentree = ['2025-08-28', '2025-08-29', '2025-08-30', '2025-08-31', '2025-09-01', '2025-09-02']
    rentree = [pd.to_datetime(d).date() for d in rentree]  
    
    return 1 if date in rentree else 0

df_axe['est_rentree'] = df_axe['Date et heure de comptage'].apply(est_rentree)

In [253]:
df_axe['est_weekend'] = df_axe['dow'].isin([5, 6]).astype(int)

In [254]:
def est_hpointe_soir(date):
    heure = date.hour
    if heure == 18 or heure == 19 :
        return 5
    return 0

df_axe['est_hpointe_soir'] = df_axe['Date et heure de comptage'].apply(est_hpointe_soir)

In [255]:
df_axe[df_axe['est_hpointe_soir']==5]['Date et heure de comptage']

1088   2024-10-01 18:00:00
1004   2024-10-01 19:00:00
1734   2024-10-02 18:00:00
1735   2024-10-02 19:00:00
550    2024-10-03 18:00:00
               ...        
794    2025-11-04 19:00:00
1647   2025-11-05 18:00:00
1646   2025-11-05 19:00:00
1891   2025-11-06 18:00:00
1892   2025-11-06 19:00:00
Name: Date et heure de comptage, Length: 728, dtype: datetime64[ns]

Nous vérifions maintenant la composition du dataset

In [256]:
df_axe = df_axe.drop('Unnamed: 0', axis = 1)
df_axe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8747 entries, 1047 to 1894
Data columns (total 43 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8747 non-null   int64         
 1   Libelle                    8747 non-null   object        
 2   Date et heure de comptage  8747 non-null   datetime64[ns]
 3   Débit horaire              4867 non-null   float64       
 4   Taux d'occupation          4974 non-null   float64       
 5   Etat trafic                8747 non-null   object        
 6   Identifiant noeud amont    8747 non-null   int64         
 7   Libelle noeud amont        8747 non-null   object        
 8   Identifiant noeud aval     8747 non-null   int64         
 9   Libelle noeud aval         8747 non-null   object        
 10  Etat arc                   8747 non-null   object        
 11  Date debut dispo data      8747 non-null   object        
 12  Date fin

In [257]:
df_axe.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,ensoleillement (en min),datetime,est_pieton,est_vacances,est_avant_vacances,est_ferie,est_avant_ferie,est_rentree,est_weekend,est_hpointe_soir
count,8747.0,8747,4867.000000,4974.000000,8747.0,8747.0,8747.000000,8747.000000,8747.000000,8747.000000,...,8747.000000,8747,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000
mean,5671.0,2025-04-26 02:38:19.028238336,398.152044,3.609356,2937.0,2973.0,11.506574,2.994741,6.668000,2024.804047,...,13.081685,2025-04-26 02:38:19.028238336,0.012804,0.356579,0.016577,0.030182,0.030182,0.016463,0.282383,0.416143
min,5671.0,2024-10-01 05:00:00,31.000000,0.000000,2937.0,2973.0,0.000000,0.000000,1.000000,2024.000000,...,0.000000,2024-10-01 05:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5671.0,2025-01-22 16:30:00,196.000000,1.319167,2937.0,2973.0,6.000000,1.000000,4.000000,2025.000000,...,0.000000,2025-01-22 16:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5671.0,2025-04-25 20:00:00,390.000000,2.626945,2937.0,2973.0,12.000000,3.000000,7.000000,2025.000000,...,0.000000,2025-04-25 20:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,5671.0,2025-08-06 21:30:00,577.000000,4.448750,2937.0,2973.0,17.000000,5.000000,10.000000,2025.000000,...,20.000000,2025-08-06 21:30:00,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,5671.0,2025-11-07 00:00:00,1024.000000,37.348340,2937.0,2973.0,23.000000,6.000000,12.000000,2025.000000,...,60.000000,2025-11-07 00:00:00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5.000000
std,0.0,NaN,224.378023,3.771356,0.0,0.0,6.919751,1.993350,3.434265,0.396955,...,21.956183,NaN,0.112436,0.479016,0.127688,0.171097,0.171097,0.127254,0.450184,1.381216


<h2>Ajout Lags</h2>

In [258]:
def process_lags(df, N_day_lags, N_last_days_forbidden=0, sampling_rate=1):
    df["Date et heure de comptage"] = pd.to_datetime(
        df["Date et heure de comptage"], utc=True
    )
    df = df.sort_values("Date et heure de comptage").set_index(
        "Date et heure de comptage"
    )

    full_index = pd.date_range(df.index.min(), df.index.max(), freq="H")
    df = df.reindex(full_index)

    Lags = []
    for lag_day in range(N_last_days_forbidden + 1, N_day_lags + 1):
        for lag_hour in range(0, 24, sampling_rate):
            lag_name = f"lag_dh_{lag_day}_{lag_hour}"
            df[lag_name] = df["Débit horaire"].shift(lag_day * 24 + lag_hour)
            Lags.append(lag_name)
            lag_name = f"lag_or_{lag_day}_{lag_hour}"
            df[lag_name] = df["Taux d'occupation"].shift(lag_day * 24 + lag_hour)
            Lags.append(lag_name)

    df = df.reset_index().rename(columns={"index": "Date et heure de comptage"})
    return df, Lags

lags = process_lags(df_axe, 35, 3, 1)
print("=== Lags ===")
print(lags[1])
df_axe = lags[0]

df_axe.tail()

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5805/465476860.py:9: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5805/465476860.py:19: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5805/465476860.py:16: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5805/465476860.py:19: PerformanceWarning:

=== Lags ===
['lag_dh_4_0', 'lag_or_4_0', 'lag_dh_4_1', 'lag_or_4_1', 'lag_dh_4_2', 'lag_or_4_2', 'lag_dh_4_3', 'lag_or_4_3', 'lag_dh_4_4', 'lag_or_4_4', 'lag_dh_4_5', 'lag_or_4_5', 'lag_dh_4_6', 'lag_or_4_6', 'lag_dh_4_7', 'lag_or_4_7', 'lag_dh_4_8', 'lag_or_4_8', 'lag_dh_4_9', 'lag_or_4_9', 'lag_dh_4_10', 'lag_or_4_10', 'lag_dh_4_11', 'lag_or_4_11', 'lag_dh_4_12', 'lag_or_4_12', 'lag_dh_4_13', 'lag_or_4_13', 'lag_dh_4_14', 'lag_or_4_14', 'lag_dh_4_15', 'lag_or_4_15', 'lag_dh_4_16', 'lag_or_4_16', 'lag_dh_4_17', 'lag_or_4_17', 'lag_dh_4_18', 'lag_or_4_18', 'lag_dh_4_19', 'lag_or_4_19', 'lag_dh_4_20', 'lag_or_4_20', 'lag_dh_4_21', 'lag_or_4_21', 'lag_dh_4_22', 'lag_or_4_22', 'lag_dh_4_23', 'lag_or_4_23', 'lag_dh_5_0', 'lag_or_5_0', 'lag_dh_5_1', 'lag_or_5_1', 'lag_dh_5_2', 'lag_or_5_2', 'lag_dh_5_3', 'lag_or_5_3', 'lag_dh_5_4', 'lag_or_5_4', 'lag_dh_5_5', 'lag_or_5_5', 'lag_dh_5_6', 'lag_or_5_6', 'lag_dh_5_7', 'lag_or_5_7', 'lag_dh_5_8', 'lag_or_5_8', 'lag_dh_5_9', 'lag_or_5_9', 'lag_d

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5805/465476860.py:16: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5805/465476860.py:19: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5805/465476860.py:16: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.conca

,Date et heure de comptage,Identifiant arc,Libelle,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,...,lag_dh_35_19,lag_or_35_19,lag_dh_35_20,lag_or_35_20,lag_dh_35_21,lag_or_35_21,lag_dh_35_22,lag_or_35_22,lag_dh_35_23,lag_or_35_23
9639,2025-11-06 20:00:00+00:00,5671.0,Convention,676.0,13.68000,Fluide,2937.0,Lecourbe-Convention,2973.0,Convention-Blomet,...,233.0,1.62167,297.0,2.03944,327.0,2.10611,370.0,2.72833,478.0,3.72667
9640,2025-11-06 21:00:00+00:00,5671.0,Convention,530.0,3.65056,Fluide,2937.0,Lecourbe-Convention,2973.0,Convention-Blomet,...,128.0,0.75722,233.0,1.62167,297.0,2.03944,327.0,2.10611,370.0,2.72833
9641,2025-11-06 22:00:00+00:00,5671.0,Convention,308.0,2.01333,Fluide,2937.0,Lecourbe-Convention,2973.0,Convention-Blomet,...,96.0,0.54778,128.0,0.75722,233.0,1.62167,297.0,2.03944,327.0,2.10611
9642,2025-11-06 23:00:00+00:00,5671.0,Convention,318.0,2.18611,Fluide,2937.0,Lecourbe-Convention,2973.0,Convention-Blomet,...,88.0,0.56667,96.0,0.54778,128.0,0.75722,233.0,1.62167,297.0,2.03944
9643,2025-11-07 00:00:00+00:00,5671.0,Convention,279.0,1.77111,Fluide,2937.0,Lecourbe-Convention,2973.0,Convention-Blomet,...,77.0,0.38500,88.0,0.56667,96.0,0.54778,128.0,0.75722,233.0,1.62167


In [259]:
df_axe.to_csv("../datasets_axes_with_all_features/" + doc , sep=";")